# Analyse ordered contact trajectories

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sthsci/Orca/blob/main/notebooks/06_trajectory_analysis.ipynb)

**Analysis notebook.** Upload binary cell histories, validate their order, inspect empirical decision states, compare four trajectory models, and export posterior results.

Run the cells from top to bottom. Values collected near the start of each notebook are safe places to experiment. Bayesian SMC fitting is deliberately disabled by default in the analysis notebooks because it can take several minutes; set `RUN_INFERENCE = True` when the data checks and descriptive plots look right.

Use synthetic or approved anonymised data only. Do not upload names, clinical metadata, raw microscopy, or a donor key that could identify participants.


## Input schema

```csv
cell_id,condition,history
cell_001,Control,"0,0,1,0"
cell_002,Control,"1,1"
cell_003,Control,""
```

`0` means an unsuccessful contact and `1` a successful contact, in experimental time order. A blank history retains a cell with zero observed contacts. `condition` is optional; the public workflow supports one to four independently fitted conditions.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def find_orca_checkout():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bayesorca").is_dir():
            return candidate
    return None


ORCA_ROOT = find_orca_checkout()
if ORCA_ROOT is not None:
    sys.path[:0] = [str(ORCA_ROOT), str(ORCA_ROOT / "src")]
elif importlib.util.find_spec("bayesorca") is None:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("ORCA currently requires a Python 3.12 Colab runtime.")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/sthsci/Orca.git@main",
        ]
    )

import bayesorca

print("bayesorca", bayesorca.__version__)
print("Python", sys.version.split()[0])


In [ ]:
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from bayesorca.trajectories import (
    TRAJECTORY_MODEL_SPECS,
    TrajectorySettings,
    TrajectorySimulationSpec,
    build_trajectory_archive,
    expanded_trajectory_frame,
    run_trajectory_conditions,
    simulate_trajectory_frame,
    trajectory_evidence_frame,
    trajectory_summary_frame,
    validate_trajectory_frame,
)

USE_UPLOAD = False
OBSERVATION_TIME = 1.0
RUN_INFERENCE = False
MODEL_KEYS = list(TRAJECTORY_MODEL_SPECS)


In [ ]:
def upload_one_csv():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Set USE_UPLOAD=True in Google Colab, or replace raw_data directly.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV file.")
    return pd.read_csv(BytesIO(next(iter(uploaded.values()))), keep_default_na=False)


if USE_UPLOAD:
    raw_data = upload_one_csv()
    truth = None
else:
    raw_data, truth = simulate_trajectory_frame(
        [
            TrajectorySimulationSpec(condition="Control", n_cells=80, seed=2026),
            TrajectorySimulationSpec(
                condition="Treatment",
                n_cells=80,
                mu_lambda=5.0,
                sigma_eta=0.45,
                beta_f=0.35,
                beta_s=-0.35,
                seed=2027,
            ),
        ]
    )

data = validate_trajectory_frame(raw_data)
print(f"Validated {len(data):,} cells across {data['condition'].nunique()} condition(s).")
data.head()


In [ ]:
per_cell = data.assign(
    contacts=data["history"].map(len),
    kills=data["history"].map(sum),
)
descriptive = (
    per_cell.groupby("condition", as_index=False)
    .agg(
        cells=("cell_id", "size"),
        mean_contacts=("contacts", "mean"),
        mean_kills=("kills", "mean"),
        zero_contact_fraction=("contacts", lambda values: values.eq(0).mean()),
    )
)
display(descriptive.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for condition, group in per_cell.groupby("condition", sort=False):
    axes[0].hist(group["contacts"], alpha=0.55, label=condition)
    axes[1].hist(group["kills"], alpha=0.55, label=condition)
axes[0].set(xlabel="Contacts per cell", ylabel="Cells")
axes[1].set(xlabel="Successful contacts per cell", ylabel="Cells")
axes[1].legend(frameon=False)
fig.suptitle("Trajectory summaries before model fitting")
fig.tight_layout()
plt.show()


In [ ]:
contacts = expanded_trajectory_frame(data)
state_summary = (
    contacts.groupby(
        ["condition", "previous_nonlethal_contacts", "previous_lethal_contacts"],
        as_index=False,
    )
    .agg(contacts=("outcome", "size"), kill_probability=("outcome", "mean"))
    .sort_values(["condition", "contacts"], ascending=[True, False])
)
state_summary.head(15)


## Fit and export

The four models cross homogeneous/heterogeneous baseline decision propensities with history-independent/history-dependent decisions. The preview uses 128 particles and 10 quadrature points; increase these and repeat independent runs before reporting model evidence.


In [ ]:
if RUN_INFERENCE:
    settings = TrajectorySettings(
        draws=128,
        chains=1,
        cores=1,
        seed=2026,
        n_quad=10,
    )
    results = run_trajectory_conditions(
        data,
        observation_time=OBSERVATION_TIME,
        settings=settings,
        model_keys=MODEL_KEYS,
    )
    display(trajectory_evidence_frame(results))
    display(trajectory_summary_frame(results))

    archive_path = Path("orca_trajectory_analysis.zip")
    archive_path.write_bytes(
        build_trajectory_archive(
            results,
            data,
            OBSERVATION_TIME,
            settings,
            truth=truth,
        )
    )
    print("Saved", archive_path.resolve())
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        pass
else:
    print("Data checks complete. Set RUN_INFERENCE = True when you are ready to fit.")


## Report with the result

Define successful/unsuccessful contacts, confirm chronological encoding, report cells and contacts per condition, observation-time units, candidate models and priors, SMC/quadrature settings, seed, ORCA version, and sensitivity runs. History coefficients describe the fitted association after the model's baseline structure; they do not alone demonstrate a causal biological memory mechanism.
